# Day 3: Ingest XML into Bronze using PySpark
Requirement: Ingest XML data into Bronze using Pyspark read and write commands.
Apply add audit columns (load_dt, source).
Create and add descriptions/metadata about enterprise data to make it more discoverable.

In [0]:
import pyspark.sql.functions as F

# Define paths and table names
source_path = "/Volumes/vstone/bronze/raw_volume/xml"
catalog = "vstone"
schema = "bronze"
bronze_table_name = "flight_xml_bronze"
bronze_table_full_name = f"{catalog}.{schema}.{bronze_table_name}"

### 1. Read XML using PySpark
Note: This requires the `com.databricks:spark-xml` library installed on the cluster.

In [0]:
try:
    raw_df = (spark.read
        .format("xml")
        .option("rowTag", "Flight") # Matching the rowTag used during data chunking
        .load(source_path)
    )

    # Add Audit Columns
    bronze_df = (raw_df
        .withColumn("load_dt", F.current_timestamp())
        .withColumn("source", F.col("_metadata.file_path"))
    )
except Exception as e:
    raise e

In [0]:
print(raw_df.count())

In [0]:
print(source_path)

### 2. Write to Delta Table (Bronze)

In [0]:
# Write to Delta table
(bronze_df.write
    .format("delta")
    .mode("append") # Append for bronze layer
    .option("mergeSchema", "true")
    .saveAsTable(bronze_table_full_name)
)

print(f"PySpark XML ingestion completed for {bronze_table_full_name}")

### 3. Add Metadata and Descriptions

In [0]:
# Add table description
spark.sql(f"COMMENT ON TABLE {bronze_table_full_name} IS 'Bronze layer table for Flight Delays ingested from XML. Contains raw data with audit columns.'")

print("Day 3: PySpark XML ingestion completed successfully.")

In [0]:
%sql
select * from vstone.bronze.flight_xml_bronze
limit 10;

In [0]:
%sql
select count(*) from vstone.bronze.flight_xml_bronze;